### **801 vs 802 vs Satellite vs NWCSAF vs Payerne Sounding Comparison**

- Compares ICON runs **801** and **802** (analyses, hourly) via total (**CLCT**) and low (**CLCL**) cloud cover, plotted side-by-side by column.
- Cross-checks model cloud cover against observations: **NWCSAF** cloud mask & low cloud type, **MSG satellite** RGB imagery, and the **Payerne radiosonde** temperature profile (lowest 1000 m).
- One figure per valid hour where all four required inputs (satellite, NWCSAF, 801, 802) are available; sounding is optional and falls back to the latest one within 24h, or shows "no data".
- Figures saved to `~/801v802vRGBvNWCvPay/`, one PNG per case; supports parallel processing across cases.

In [2]:
import os
os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"

SAVE_FIGURES = True 
SHOW_FIGURES = False
PARALLEL = True   # only takes effect when SAVE_FIGURES=True
MAX_WORKERS = 8

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib

if SAVE_FIGURES:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm

from datetime import datetime, timedelta
from pathlib import Path
from PIL import Image

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

import earthkit.data as ekd

# ==========================================================
# SETTINGS
# ==========================================================

SEARCH_START = datetime(2025, 10, 4, 0)
SEARCH_END = datetime(2025, 10, 10, 23)

NWCSAF_DIR = "/scratch/mch/jdelbeke/nwcsaf"
NWCSAF_NAME_FMT = "MSG_nwcsaf_cosmo1eqc3km_{:%Y%m%d%H%M}.nc"

SAT_DIR = os.path.expanduser("~/Raw_data/satellite_raw")
SAT_NAME_FMT = "MSG_RGB-DayNightFog_cosmo1_{:%Y%m%d%H%M}.png"

# analyses
ICON_801_DIR = "/store_new/mch/msopr/jdelbeke/ICON_TST/801/FG25/det"
ICON_802_DIR = "/store_new/mch/msopr/jdelbeke/ICON_TST/802/FG25/det"
ICON_NAME_FMT = "lff{:%Y%m%d%H}"

CLC_CMAP = "Blues"

SOUNDING_DIR = Path("/scratch/mch/jdelbeke/soundings/output/")
SOUNDING_STATION_ID = "06610"
SOUNDING_MAX_HEIGHT_M = 1000
SOUNDING_MAX_AGE_HOURS = 24

# NWCSAF cloud type categories
CT_KEEP = [5, 6]
CT_LABELS = {5: "Very low clouds", 6: "Low clouds"}

OUTPUT_DIR = os.path.expanduser("~/801v802vRGBvNWCvPay")
if SAVE_FIGURES:
    os.makedirs(OUTPUT_DIR, exist_ok=True)


# ==========================================================
# FIND FILES
# ==========================================================

def find_sat_file(valid_time):
    path = os.path.join(SAT_DIR, SAT_NAME_FMT.format(valid_time))
    return path if os.path.exists(path) else None

def find_nwcsaf_file(valid_time):
    path = os.path.join(NWCSAF_DIR, NWCSAF_NAME_FMT.format(valid_time))
    return path if os.path.exists(path) else None

def find_icon_analysis(base_dir, valid_time):
    path = os.path.join(base_dir, ICON_NAME_FMT.format(valid_time))
    return path if os.path.exists(path) else None

def find_sounding_file(valid_time):
    path = SOUNDING_DIR / f"{valid_time:%Y%m%d}_{SOUNDING_STATION_ID}_data.txt"
    return path if path.exists() else None


# ==========================================================
# READERS
# ==========================================================

def load_icon_fieldlist(grib_file):
    return ekd.from_source("file", grib_file).to_fieldlist()

def extract_field(fields, param):
    xa = fields.sel({"parameter.variable": param})[0].to_xarray()
    return xa["longitude"].values, xa["latitude"].values, xa[param].values

def read_sounding(file):
    with open(file) as f:
        header = f.readlines()[1].split()
    return pd.read_csv(
        file, sep=r"\s*\|\s*", engine="python",
        skiprows=3, header=None, names=header,
    )

def load_sounding_day(day_dt):
    """Load one day's sounding file with a parsed termin_dt column, or None."""
    file = find_sounding_file(day_dt)
    if file is None:
        return None

    df = read_sounding(file)
    try:
        df["termin"] = df["termin"].astype(int)
    except (KeyError, ValueError):
        return None

    df["termin_dt"] = pd.to_datetime(df["termin"].astype(str), format="%Y%m%d%H%M%S")
    return df

def get_latest_payerne_profile(valid_time, max_age_hours=SOUNDING_MAX_AGE_HOURS,
                                max_height=SOUNDING_MAX_HEIGHT_M):
    earliest = valid_time - timedelta(hours=max_age_hours)

    frames = [
        df for offset in range(int(max_age_hours // 24) + 2)
        if (df := load_sounding_day(valid_time - timedelta(days=offset))) is not None
    ]
    if not frames:
        return None

    combined = pd.concat(frames, ignore_index=True)
    window = combined[(combined["termin_dt"] <= valid_time) & (combined["termin_dt"] >= earliest)]
    if window.empty:
        return None

    sounding_time = window["termin_dt"].max()
    sounding = window[window["termin_dt"] == sounding_time]

    profile = (
        sounding[["742", "745"]]
        .replace(10000000, np.nan)
        .dropna()
        .rename(columns={"742": "height", "745": "temperature"})
        .sort_values("height")
    )
    if len(profile) < 2:
        return None

    # convert absolute altitude to height above first valid observation
    profile["height"] -= profile["height"].iloc[0]
    profile = profile[profile["height"] <= max_height]
    if len(profile) < 2:
        return None

    profile = profile.groupby("height", as_index=False)["temperature"].mean().sort_values("height")
    return profile, sounding_time.to_pydatetime()
    

# ==========================================================
# DISCOVER CASES (every hour, analyses)
# ==========================================================

def collect_cases():
    cases = []
    t = SEARCH_START
    while t <= SEARCH_END:
        files = {
            "sat_file": find_sat_file(t),
            "nwcsaf_file": find_nwcsaf_file(t),
            "icon801_file": find_icon_analysis(ICON_801_DIR, t),
            "icon802_file": find_icon_analysis(ICON_802_DIR, t),
        }
        if all(files.values()):
            cases.append({"valid_time": t, **files})
        t += timedelta(hours=1)
    return cases


# ==========================================================
# PROCESS ONE CASE
# ==========================================================

def process_case(case):
    valid_time = case["valid_time"]

    # --- NWCSAF (cloud mask + cloud type share the same file/grid) ---
    ds = xr.open_dataset(case["nwcsaf_file"])
    cma = ds["cma"].where(ds["cma"] != 255)
    ct = ds["ct"].where(ds["ct"].isin(CT_KEEP)).fillna(0)
    lon_nwc, lat_nwc = ds["longitude"].values, ds["latitude"].values

    # --- satellite image ---
    sat_img = Image.open(case["sat_file"]).convert("RGB")

    # --- ICON 801 / 802 (open each grib once, extract both params) ---
    fields_801 = load_icon_fieldlist(case["icon801_file"])
    lon_801, lat_801, clct_801 = extract_field(fields_801, "CLCT")
    _, _, clcl_801 = extract_field(fields_801, "CLCL")

    fields_802 = load_icon_fieldlist(case["icon802_file"])
    lon_802, lat_802, clct_802 = extract_field(fields_802, "CLCT")
    _, _, clcl_802 = extract_field(fields_802, "CLCL")

    # --- optional Payerne sounding: latest available at or before valid_time ---
    sounding_result = get_latest_payerne_profile(valid_time)
    profile, sounding_time = sounding_result if sounding_result else (None, None)

    # ------------------------------------------------------
    # PLOT
    # ------------------------------------------------------
    fig = plt.figure(figsize=(18, 14))
    gs = fig.add_gridspec(
        nrows=3, ncols=3,
        left=0.05, right=0.82, top=0.90, bottom=0.08,
        hspace=-0.25, wspace=0.20,
    )

    ax_801_clct = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
    ax_802_clct = fig.add_subplot(gs[1, 0], projection=ccrs.PlateCarree())
    ax_801_clcl = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())
    ax_802_clcl = fig.add_subplot(gs[1, 1], projection=ccrs.PlateCarree())
    ax_sounding = fig.add_subplot(gs[0:2, 2])  # spans first two rows of col 3

    ax_nwc_cma = fig.add_subplot(gs[2, 0], projection=ccrs.PlateCarree())
    ax_nwc_ct = fig.add_subplot(gs[2, 1], projection=ccrs.PlateCarree())
    ax_sat = fig.add_subplot(gs[2, 2])

    # narrow the sounding panel and shift it right
    pos = ax_sounding.get_position()
    ax_sounding.set_position([pos.x0 + 0.09, pos.y0, pos.width * 0.55, pos.height])
    ax_sounding.tick_params(labelsize=8)
    ax_sounding.set_box_aspect(1.8)

    # panels 1-4: 801 / 802 CLCT & CLCL
    sc_801_clct = ax_801_clct.scatter(lon_801, lat_801, c=clct_801, s=2, cmap=CLC_CMAP, vmin=0, vmax=100, transform=ccrs.PlateCarree())
    ax_801_clct.set_title(f"801 CLCT\n{valid_time:%Y-%m-%d %H:%M UTC}")
    ax_802_clct.scatter(lon_802, lat_802, c=clct_802, s=2, cmap=CLC_CMAP, vmin=0, vmax=100, transform=ccrs.PlateCarree())
    ax_802_clct.set_title(f"802 CLCT\n{valid_time:%Y-%m-%d %H:%M UTC}")
    ax_801_clcl.scatter(lon_801, lat_801, c=clcl_801, s=2, cmap=CLC_CMAP, vmin=0, vmax=100, transform=ccrs.PlateCarree())
    ax_801_clcl.set_title(f"801 CLCL\n{valid_time:%Y-%m-%d %H:%M UTC}")
    sc_802_clcl = ax_802_clcl.scatter(lon_802, lat_802, c=clcl_802, s=2, cmap=CLC_CMAP, vmin=0, vmax=100, transform=ccrs.PlateCarree())
    ax_802_clcl.set_title(f"802 CLCL\n{valid_time:%Y-%m-%d %H:%M UTC}")

    # panel 5: sounding, lowest SOUNDING_MAX_HEIGHT_M meters
    if profile is not None:
        ax_sounding.plot(profile["temperature"], profile["height"], marker="o", markersize=3)
        ax_sounding.set_xlabel("Temperature (°C)")
        ax_sounding.set_ylabel("Height above station (m)")
        ax_sounding.set_ylim(0, SOUNDING_MAX_HEIGHT_M)
        ax_sounding.grid(True, alpha=0.4)
    else:
        ax_sounding.text(0.5, 0.5, "No sounding data", ha="center", va="center",
                          transform=ax_sounding.transAxes, fontsize=11, color="0.4")
        ax_sounding.set_xticks([])
        ax_sounding.set_yticks([])

    if sounding_time is not None:
        age_h = (valid_time - sounding_time).total_seconds() / 3600
        ax_sounding.set_title(f"Payerne sounding\n{sounding_time:%Y-%m-%d %H:%M UTC}\n({age_h:.0f}h before valid time)")
    else:
        ax_sounding.set_title(f"Payerne sounding {valid_time:%Y-%m-%d %H:%M UTC}")

    # panel 6: NWCSAF binary cloud mask
    cma_cmap = ListedColormap(["#f0f0f0", "#3b6ea5"])
    cma_norm = BoundaryNorm([-0.5, 0.5, 1.5], cma_cmap.N)
    ax_nwc_cma.pcolormesh(lon_nwc, lat_nwc, cma, shading="auto", cmap=cma_cmap, norm=cma_norm, transform=ccrs.PlateCarree())
    ax_nwc_cma.set_title(f"NWCSAF cloud mask\n{valid_time:%Y-%m-%d %H:%M UTC}")
    ax_nwc_cma.legend(
        handles=[
            mpatches.Patch(color="#f0f0f0", label="Cloud-free", ec="0.5"),
            mpatches.Patch(color="#3b6ea5", label="Cloudy"),
        ],
        loc="lower left", framealpha=.9, fontsize=8,
    )

    # panel 7: NWCSAF low cloud types only
    ct_cmap = ListedColormap(["#ffffff", "#c7e9c0", "#74c476"])  # none / very low / low
    ct_norm = BoundaryNorm([-0.5, 0.5, 5.5, 6.5], ct_cmap.N)
    sc_ct = ax_nwc_ct.pcolormesh(lon_nwc, lat_nwc, ct, shading="auto", cmap=ct_cmap, norm=ct_norm, transform=ccrs.PlateCarree())
    ax_nwc_ct.set_title(f"NWCSAF low cloud type\n{valid_time:%Y-%m-%d %H:%M UTC}")
    ax_nwc_ct.legend(
        handles=[
            mpatches.Patch(color="#c7e9c0", label="Very low clouds"),
            mpatches.Patch(color="#74c476", label="Low clouds"),
        ],
        loc="lower left", framealpha=.9, fontsize=8,
    )

    # panel 8: satellite
    ax_sat.imshow(sat_img)
    ax_sat.axis("off")

    # map settings for all geo panels
    for ax in [ax_801_clct, ax_802_clct, ax_801_clcl, ax_802_clcl, ax_nwc_cma, ax_nwc_ct]:
        ax.set_extent([-2, 19, 41, 52], crs=ccrs.PlateCarree())
        ax.coastlines(resolution="10m", linewidth=0.8)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    
        xlocs = list(range(-2, 20, 5))
        ylocs = list(range(41, 53, 2))
    
        ax.gridlines(xlocs=xlocs, ylocs=ylocs, linewidth=0.3, alpha=0.5)
        ax.set_xticks(xlocs, crs=ccrs.PlateCarree())
        ax.set_yticks(ylocs, crs=ccrs.PlateCarree())
        ax.xaxis.set_major_formatter(LongitudeFormatter())
        ax.yaxis.set_major_formatter(LatitudeFormatter())
        ax.tick_params(labelsize=7)

    # shared colorbar for the 4 continuous cloud-cover panels (rows 1-2)
    cax_clc = fig.add_axes([0.56, 0.45, 0.008, 0.32])
    fig.colorbar(sc_802_clcl, cax=cax_clc).set_label("Cloud cover (%)")

    if SAVE_FIGURES:
        out_path = os.path.join(OUTPUT_DIR, f"Sat_NWCSAF_ICON_Pay_{valid_time:%Y%m%d_%H%M}.png")
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        print("Saved:", out_path)

    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)


# ==========================================================
# MAIN
# ==========================================================

if __name__ == "__main__":
    cases = collect_cases()
    if not cases:
        raise FileNotFoundError("No matching sat/NWCSAF/801/802 files found")
    print(f"Found {len(cases)} cases")

    if PARALLEL and SAVE_FIGURES:
        import concurrent.futures
        with concurrent.futures.ProcessPoolExecutor(max_workers=MAX_WORKERS) as pool:
            list(pool.map(process_case, cases))
        n_processed = len(cases)
    else:
        for n_processed, case in enumerate(cases, start=1):
            print(f"[{n_processed}/{len(cases)}] {case['valid_time']:%Y-%m-%d %H:%M}")
            process_case(case)

    print(f"\nDone: {n_processed} frame(s) plotted.")

Found 143 cases
Saved: /users/jdelbeke/801v802vRGBvNWCvPay/Sat_NWCSAF_ICON_Pay_20251004_0200.png
Saved: /users/jdelbeke/801v802vRGBvNWCvPay/Sat_NWCSAF_ICON_Pay_20251004_0600.png
Saved: /users/jdelbeke/801v802vRGBvNWCvPay/Sat_NWCSAF_ICON_Pay_20251004_0400.png
Saved: /users/jdelbeke/801v802vRGBvNWCvPay/Sat_NWCSAF_ICON_Pay_20251004_0100.png
Saved: /users/jdelbeke/801v802vRGBvNWCvPay/Sat_NWCSAF_ICON_Pay_20251004_0800.png
Saved: /users/jdelbeke/801v802vRGBvNWCvPay/Sat_NWCSAF_ICON_Pay_20251004_0300.png
Saved: /users/jdelbeke/801v802vRGBvNWCvPay/Sat_NWCSAF_ICON_Pay_20251004_0500.png
Saved: /users/jdelbeke/801v802vRGBvNWCvPay/Sat_NWCSAF_ICON_Pay_20251004_0700.png
Saved: /users/jdelbeke/801v802vRGBvNWCvPay/Sat_NWCSAF_ICON_Pay_20251004_1000.png
Saved: /users/jdelbeke/801v802vRGBvNWCvPay/Sat_NWCSAF_ICON_Pay_20251004_0900.png
Saved: /users/jdelbeke/801v802vRGBvNWCvPay/Sat_NWCSAF_ICON_Pay_20251004_1200.png
Saved: /users/jdelbeke/801v802vRGBvNWCvPay/Sat_NWCSAF_ICON_Pay_20251004_1500.png
Saved: /user